# Tutorial 1a: Variables and Evolution

Tutorial 1 introduced `laila.constant` and `laila.variable`. This tutorial digs into the second one — the `evolution` counter that lets a single nickname address an ordered sequence of versions of the same logical entry.

You will learn:

- What `evolution` is and how it shapes a `global_id`
- How to recall a specific version with `remember(nickname=..., evolution=N)`
- How `Entry.evolve(new_data=...)` produces the next version
- Why `laila.constant` cannot be evolved

**Prerequisites:** `pip install laila-core` — no credentials or external services required.

In [ ]:
import laila
from laila.macros.defaults import DefaultPool

laila.memory.extend(DefaultPool(), pool_nickname="evo")

## Constant vs. variable global IDs

A `constant` has `evolution = None` and no suffix on its `global_id`. A `variable` starts at `evolution = 0` (or whatever you pass) and carries a `-N` suffix derived from the counter.

In [ ]:
c = laila.constant(data="immutable", nickname="model.config")
v = laila.variable(data=[0.1, 0.2, 0.3], nickname="model.weights")

print("constant:", c.global_id, " evolution =", c.evolution)
print("variable:", v.global_id, " evolution =", v.evolution)

The trailing `-0` on the variable's `global_id` is what makes evolutions addressable — two entries with the same nickname but different evolutions are two **different keys** in any pool.

## Memorize and recall a specific evolution

`laila.remember(nickname=..., evolution=N)` derives the same `global_id` you saw above. Pass the evolution explicitly to round-trip a single version:

In [ ]:
laila.memorize(v, pool_nickname="evo").wait()

recovered = laila.remember(nickname="model.weights", evolution=0, pool_nickname="evo").wait()
print("recovered:", recovered.data)

## Evolving — bumping the version

`Entry.evolve(new_data=...)` returns a **new** entry that shares the variable's UUID but has `evolution + 1`. The original entry is **not** mutated — you must rebind to capture the new version.

In [ ]:
history = [v]
for step in range(1, 5):
    next_v = history[-1].evolve(data=[x + 0.1 for x in history[-1].data])
    laila.memorize(next_v, pool_nickname="evo").wait()
    history.append(next_v)

for h in history:
    print(h.global_id, h.data)

All five evolutions now live in the pool as distinct keys. You can recall any of them by passing `evolution=N`:

In [ ]:
for i in range(5):
    e = laila.remember(nickname="model.weights", evolution=i, pool_nickname="evo").wait()
    print(f"evolution {i}: {e.data}")

## Constants cannot evolve

`Entry.evolve` is defined on variables only. Calling it on a constant raises `RuntimeError`:

In [ ]:
try:
    c.evolve(data="oops")
except (RuntimeError, AttributeError) as e:
    print("caught:", type(e).__name__, "-", e)

## When to use which

| Use a `constant` for | Use a `variable` for |
|---|---|
| Configs, hyperparameters, dataset splits | Model weights, training metrics |
| Reference data that should never change | Anything you'll re-write under the same logical name |
| Source-of-truth content addressed by nickname | Ordered history you want to walk by version |

A practical rule: if you would ever want to look at "the entry I produced yesterday", reach for a variable so yesterday's evolution is still on disk after today's update.

## Summary

- A variable's `global_id` ends in `-N` where `N` is its evolution counter.
- `remember(nickname=..., evolution=N)` selects a specific version.
- `entry.evolve(new_data=...)` returns a new entry with `evolution + 1` — assign it back.
- Constants are immutable; `evolve` raises.

Next: continue with [Tutorial 2 — Local Pools](02_local_pools.ipynb) to see how variables interact with the three local storage backends.